<a href="https://colab.research.google.com/github/muhammadusmanshakir/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\nRequired columns:")
print([
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
])

display(df.head())

Rows: 30,000
Columns: 44

Required columns:
['content_id', 'client_id', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks and rule reasoning

### Signal 1 — Staleness

**Signal:** `days_since_last_update`

**Why I chose it:** Staleness is directly related to FlyRank's refresh-flag logic. My hypothesis is that older content should receive greater consideration for review, particularly when it still has meaningful search visibility.

I will inspect the distribution using four buckets:

- `<30 days`
- `30–89 days`
- `90–179 days`
- `180+ days`

The verdict will be based on the observed data rather than an assumed relationship.

In [ ]:
# Signal 1: Staleness audit

staleness_bins = [-np.inf, 29, 89, 179, np.inf]
staleness_labels = [
    "<30 days",
    "30–89 days",
    "90–179 days",
    "180+ days"
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          mean_impressions_90d=("impressions_90d", "mean"),
          median_impressions_90d=("impressions_90d", "median")
      )
      .reset_index()
)

staleness_table["share"] = (
    staleness_table["n"] / len(df)
)

print("STALENESS SIGNAL AUDIT")
print(staleness_table.to_string(index=False))

print("\nTotal n:", len(df))


STALENESS SIGNAL AUDIT
staleness_bucket     n  mean_impressions_90d  median_impressions_90d    share
        <30 days 20480           4199.614062                   470.0 0.682667
      30–89 days   175           6506.748571                   510.0 0.005833
     90–179 days  9171           7486.665140                  1692.0 0.305700
       180+ days   174           1172.448276                    15.5 0.005800

Total n: 30000


### Signal 2 — Search Visibility

**Signal:** `impressions_90d`

**Why I chose it:** Search impressions provide a transparent measure of observed visibility. A stale page with meaningful search visibility may represent a more useful optimization opportunity than a page with almost no observed visibility.

I will inspect four buckets:

- `<100`
- `100–999`
- `1,000–4,999`
- `5,000+`

This signal supports the decision to prioritize stale pages that still have observable search exposure.

In [ ]:
# Signal 2: Search visibility audit

impression_bins = [-np.inf, 99, 999, 4999, np.inf]

impression_labels = [
    "<100",
    "100–999",
    "1,000–4,999",
    "5,000+"
]

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=impression_bins,
    labels=impression_labels
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          mean_staleness=("days_since_last_update", "mean"),
          median_staleness=("days_since_last_update", "median")
      )
      .reset_index()
)

visibility_table["share"] = (
    visibility_table["n"] / len(df)
)

print("SEARCH VISIBILITY SIGNAL AUDIT")
print(visibility_table.to_string(index=False))

print("\nTotal n:", len(df))

SEARCH VISIBILITY SIGNAL AUDIT
visibility_bucket    n  mean_staleness  median_staleness    share
             <100 7994       33.120090              20.0 0.266467
          100–999 8494       46.369908              22.0 0.283133
      1,000–4,999 7361       51.147398              22.0 0.245367
           5,000+ 6151       56.547716              25.0 0.205033

Total n: 30000


## Signal Verdicts

### 1. Staleness — CONFIRMED

The staleness signal is useful for the baseline because pages in the
90–179 day bucket have higher average and median impressions than
pages updated within the last 30 days. The 180+ day bucket is small
(n = 174), so it should be treated cautiously rather than assumed to
represent all stale content.

**Verdict: CONFIRMED**

### 2. Search Visibility — CONFIRMED

Search visibility is also useful for the baseline. As impressions
increase across the buckets, mean staleness increases from about
33 days for pages with fewer than 100 impressions to about 57 days
for pages with 5,000+ impressions. This suggests that visible pages
can still contain older content worth reviewing.

**Verdict: CONFIRMED**

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline rule

I will use a simple, transparent rule based on two observed signals: content staleness and search visibility.

A page receives a higher score when it has not been updated for a long time and has accumulated substantial search impressions. The idea is that older pages with meaningful search visibility have a larger potential opportunity for review.

The rule uses:
- `days_since_last_update` as the staleness signal.
- `impressions_90d` as the visibility signal.

The rule does not use `trend_pct`, `trend_direction`, or any label-derived information.

Reason codes:
- `stale_and_visible` — the page is both stale and visibly active in search.
- `stale` — the page is stale but has lower visibility.
- `visible` — the page has meaningful visibility but is not yet stale.
- `low_priority` — neither condition is strongly present.

Action labels:
- `Review content` — prioritize manual content review.
- `Monitor` — keep under observation.
- `No immediate action` — lower priority for this baseline.

In [ ]:
# ============================================
# SECTION 2 — BUILD THE RANKED BASELINE QUEUE
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np

print(f"Rows loaded: {len(df):,}")

# --------------------------------------------
# 1. Define transparent baseline signals
# --------------------------------------------

stale = df["days_since_last_update"] >= 90
visible = df["impressions_90d"] >= 1000

# --------------------------------------------
# 2. Build baseline score
# --------------------------------------------

df["baseline_score"] = (
    stale.astype(int)
    * visible.astype(int)
    * df["impressions_90d"]
)

# --------------------------------------------
# 3. Reason code
# --------------------------------------------

df["reason_code"] = np.where(
    stale & visible,
    "stale_and_visible",
    "not_priority"
)

# --------------------------------------------
# 4. Action
# --------------------------------------------

df["action"] = np.where(
    stale & visible,
    "Review content",
    "Monitor"
)

# --------------------------------------------
# 5. Rank the queue
# --------------------------------------------

queue = (
    df[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr"
        ]
    ]
    .sort_values(
        by="baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

print("\nBaseline queue created successfully.")
print(f"Rows in queue: {len(queue):,}")

display(queue.head(10))

# --------------------------------------------
# 6. Write the ranked queue to CSV (required deliverable)
# --------------------------------------------

output_path = Path("work/outputs")
output_path.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path / "baseline_action_score.csv", index=False)

print(f"\nSaved ranked queue to {output_path / 'baseline_action_score.csv'}")
print(f"Rows written: {len(queue):,}")


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 Review

I reviewed the highest-ranked items from the baseline queue.

The baseline action is `refresh_content` when a content item is both stale
(at least 90 days since the last update) and visible (at least 1,000
impressions in the last 90 days).

The score prioritizes the highest-impression stale pages first.

For each item, I record the action, why the baseline selected it, and what
could make the recommendation wrong. These are directional decision-support
judgments, not guarantees that a page should be refreshed.

In [ ]:
# ============================================
# SECTION 3 — TOP-20 SKEPTIC REVIEW
# ============================================

top20 = queue.head(20).copy()

def review_row(row):
    if row["action"] == "Review content":
        action = "Review content"
        why = (
            f"Stale ({int(row['days_since_last_update'])} days) "
            f"and highly visible ({int(row['impressions_90d']):,} impressions)."
        )
        wrong = (
            "Could be wrong if the page is intentionally evergreen, "
            "already scheduled for refresh, or its traffic is not strategically valuable."
        )
    else:
        action = "Monitor"
        why = "Does not meet both baseline priority conditions."
        wrong = (
            "Could be wrong if other business or search signals make the page "
            "important despite its low baseline score."
        )

    return pd.Series({
        "action": action,
        "why_it_is_here": why,
        "what_would_make_it_wrong": wrong
    })

review = top20.apply(review_row, axis=1)

top20_review = pd.concat(
    [
        top20[
            [
                "rank",
                "content_id",
                "client_id",
                "baseline_score",
                "days_since_last_update",
                "impressions_90d",
                "avg_position",
                "ctr"
            ]
        ].reset_index(drop=True),
        review.reset_index(drop=True)
    ],
    axis=1
)

display(top20_review)


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Section 4: Weak picks + leakage check

# Show potentially weak picks among the top 20
weak_picks = top20_review[
    (top20_review["avg_position"] > 20) |
    (top20_review["ctr"] == 0)
].copy()

print("Potential weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "avg_position",
            "ctr",
            "action",
            "why_it_is_here",
            "what_would_make_it_wrong",
        ]
    ]
)

print("\nLeakage check:")

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_columns = set(df.columns)

for col in forbidden_columns:
    print(f"{col}: {'PRESENT IN DATA' if col in used_columns else 'NOT PRESENT'}")

print("\nRule features actually used:")
print([
    "days_since_last_update",
    "impressions_90d"
])

print("\nConclusion:")
print(
    "The baseline rule uses only current/historical snapshot signals "
    "and does not use trend_direction, trend_pct, or label-derived inputs."
)

Potential weak picks:


,rank,content_id,baseline_score,avg_position,ctr,action,why_it_is_here,what_would_make_it_wrong
1,2,content_2dba2b1f9536,443434,27.9,0.21,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...
6,7,content_b28d1efd668f,286608,26.2,0.06,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...
7,8,content_813e88069237,233561,26.2,0.06,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...
9,10,content_c8e9d6ab9013,208678,9.7,0.00,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...
10,11,content_b511d4bc4ad2,205915,27.9,0.14,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...
17,18,content_f02b48f88241,181514,25.8,0.10,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...
18,19,content_05e9b4cd9ccf,179002,22.1,0.08,monitor,Does not meet both baseline priority conditions.,Could be wrong if other business or search sig...



Leakage check:
trend_direction: PRESENT IN DATA
trend_pct: PRESENT IN DATA
is_declining_label: NOT PRESENT

Rule features actually used:
['days_since_last_update', 'impressions_90d']

Conclusion:
The baseline rule uses only current/historical snapshot signals and does not use trend_direction, trend_pct, or label-derived inputs.


## 4. Weak Picks + Leakage Check

The baseline produced several potentially weak recommendations.

Some top-ranked pages have relatively poor average positions, while one has zero CTR.
This does not automatically mean the recommendation is wrong because the baseline
intentionally prioritizes stale pages with high visibility.

These weak picks show an important limitation of the baseline: high impressions alone
do not prove that refreshing a page is the best action.

The rule uses only `days_since_last_update` and `impressions_90d`.
Although `trend_direction` and `trend_pct` exist in the dataset, they were not used
as rule inputs. No future-window or label-derived inputs were used.

The result is therefore treated as directional decision-support rather than a guarantee
that a page should be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.